# YgoMaster-PackBuilder — Rarity Mapping Tests

Notebook di test manuale/esplorativo per il rarity normalization layer.

Obiettivi:

- caricare `config/rarity_mapping.yaml`;
- verificare la conversione verso `domain.Rarity`;
- normalizzare una singola `Card`;
- normalizzare una lista di carte in place;
- verificare che `source_rarity` venga preservata;
- verificare la case sensitivity;
- verificare il comportamento con rarità non mappate;
- verificare mapping YAML non validi.


## 1. Ambiente e root repository

In [6]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository."
    )

mapping_path = repo_root / "config" / "rarity_mapping.yaml"

assert mapping_path.exists(), (
    f"Rarity mapping non trovato: {mapping_path}"
)

print("Repository root:", repo_root)
print("Mapping path:", mapping_path)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder
Mapping path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\rarity_mapping.yaml


## 2. Import

In [7]:
from copy import deepcopy
from tempfile import TemporaryDirectory

from ygomaster_packbuilder.domain import Card, Rarity

from ygomaster_packbuilder.mapping import (
    load_rarity_mapping,
    normalize_card_rarity,
    normalize_cards,
)

print("Imports OK")

Imports OK


## 3. Caricamento del mapping reale

In [8]:
mapping = load_rarity_mapping(mapping_path)

assert isinstance(mapping, dict)

print(mapping)

{'Common': <Rarity.N: 'N'>, 'Rare': <Rarity.R: 'R'>, 'Super Rare': <Rarity.SR: 'SR'>, 'Ultra Rare': <Rarity.UR: 'UR'>}


## 4. Mapping atteso

In [9]:
assert mapping["Common"] is Rarity.N
assert mapping["Rare"] is Rarity.R
assert mapping["Super Rare"] is Rarity.SR
assert mapping["Ultra Rare"] is Rarity.UR

print("Base rarity mapping OK")

Base rarity mapping OK


## 5. Tipi dei valori

In [10]:
for source_name, rarity in mapping.items():
    assert isinstance(source_name, str)
    assert isinstance(rarity, Rarity)

print("All mapping values are domain.Rarity instances")

All mapping values are domain.Rarity instances


## 6. Normalizzazione di una singola carta

In [11]:
card = Card(
    name='"A" Cell Breeding Device',
    ydk_id=34541863,
    source_rarity="Common",
)

assert card.rarity is None

normalize_card_rarity(card, mapping)

assert card.rarity is Rarity.N
assert card.source_rarity == "Common"

print(card)

Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=None, rarity=<Rarity.N: 'N'>)


## 7. Normalizzazione in place

La funzione deve modificare lo stesso oggetto, non crearne uno nuovo.


In [12]:
card = Card(
    name="Alien Ammonite",
    ydk_id=91123920,
    source_rarity="Rare",
)

original_id = id(card)

normalize_card_rarity(card, mapping)

assert id(card) == original_id
assert card.rarity is Rarity.R

print("In-place normalization OK")

In-place normalization OK


## 8. Normalizzazione di una lista

In [13]:
cards = [
    Card(
        name="Common Test",
        ydk_id=1,
        source_rarity="Common",
    ),
    Card(
        name="Rare Test",
        ydk_id=2,
        source_rarity="Rare",
    ),
    Card(
        name="Super Rare Test",
        ydk_id=3,
        source_rarity="Super Rare",
    ),
    Card(
        name="Ultra Rare Test",
        ydk_id=4,
        source_rarity="Ultra Rare",
    ),
]

normalize_cards(cards, mapping)

expected = [
    Rarity.N,
    Rarity.R,
    Rarity.SR,
    Rarity.UR,
]

assert [card.rarity for card in cards] == expected

for card in cards:
    print(card.name, "->", card.rarity)

Common Test -> N
Rare Test -> R
Super Rare Test -> SR
Ultra Rare Test -> UR


## 9. `source_rarity` viene preservata

In [14]:
cards = [
    Card(
        name="Test",
        ydk_id=123,
        source_rarity="Super Rare",
    )
]

normalize_cards(cards, mapping)

assert cards[0].source_rarity == "Super Rare"
assert cards[0].rarity is Rarity.SR

print(cards[0])

Card(name='Test', ydk_id=123, source_rarity='Super Rare', master_duel_id=None, rarity=<Rarity.SR: 'SR'>)


## 10. `master_duel_id` non viene toccato

La normalizzazione della rarità non deve occuparsi della risoluzione degli ID.


In [15]:
card = Card(
    name="Test",
    ydk_id=123,
    source_rarity="Ultra Rare",
)

assert card.master_duel_id is None

normalize_card_rarity(card, mapping)

assert card.rarity is Rarity.UR
assert card.master_duel_id is None

print("Master Duel ID untouched")

Master Duel ID untouched


## 11. Rarità non mappata: `Secret Rare`

Finché non decidiamo esplicitamente come mapparla, deve essere rifiutata.


In [16]:
card = Card(
    name="Secret Rare Test",
    ydk_id=999,
    source_rarity="Secret Rare",
)

try:
    normalize_card_rarity(card, mapping)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "Secret Rare avrebbe dovuto essere rifiutata."
    )

assert card.rarity is None

Expected ValueError: Unsupported source rarity: 'Secret Rare'.


## 12. Case sensitivity

Il mapping è intenzionalmente case-sensitive.


In [17]:
card = Card(
    name="Case Test",
    ydk_id=1000,
    source_rarity="common",
)

try:
    normalize_card_rarity(card, mapping)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "'common' non dovrebbe essere equivalente a 'Common'."
    )

Expected ValueError: Unsupported source rarity: 'common'.


## 13. Lista vuota

In [18]:
cards = []

result = normalize_cards(cards, mapping)

assert cards == []

print("Empty list normalization OK")
print("Return value:", result)

Empty list normalization OK
Return value: None


## 14. Mapping YAML con valore domain non valido

Esempio:

```yaml
rarities:
  Common: "INVALID"
```

deve essere rifiutato dal loader.


In [19]:
with TemporaryDirectory() as tmpdir:
    invalid_path = Path(tmpdir) / "invalid_mapping.yaml"

    invalid_path.write_text(
        '''
rarities:
  Common: "INVALID"
'''.strip(),
        encoding="utf-8",
    )

    try:
        load_rarity_mapping(invalid_path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Il loader avrebbe dovuto rifiutare Rarity='INVALID'."
        )

Expected ValueError: Invalid rarity value for key 'Common': INVALID


## 15. File inesistente

In [20]:
missing = repo_root / "config" / "__missing_rarity_mapping__.yaml"

try:
    load_rarity_mapping(missing)
except FileNotFoundError as exc:
    print("Expected FileNotFoundError:", exc)
else:
    raise AssertionError(
        "Il loader avrebbe dovuto sollevare FileNotFoundError."
    )

Expected FileNotFoundError: Rarity mapping file not found: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\__missing_rarity_mapping__.yaml


## 16. Root YAML non mapping

In [21]:
with TemporaryDirectory() as tmpdir:
    invalid_path = Path(tmpdir) / "invalid_root.yaml"

    invalid_path.write_text(
        "- Common\n- Rare\n",
        encoding="utf-8",
    )

    try:
        load_rarity_mapping(invalid_path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Il loader avrebbe dovuto rifiutare un root YAML non-mapping."
        )

Expected ValueError: Rarity mapping root must be a mapping.
